

# FAA SAIB RAG Agent
# Grounded question-answering over FAA Special Airworthiness Information Bulletins (SAIBs),
# built Spark-native on Databricks with Vector Search and Llama 3.3 70B.

# **Pipeline:** PDFs (volume) → parse → extract + clean IDs → Delta tables →
# Vector Search index → retrieval + LLM agent → evaluation.
 

In [0]:
%pip install databricks-vectorsearch

Looking in indexes: [REDACTED]
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6acf96d4-7357-4d50-8d0d-922d3b5c180c
    Can't uninstall 'protobuf'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [databricks-vectorsearch]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.76.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# %restart_python

In [0]:
VOLUME_PATH   = "/Volumes/demo_projects/default/dynamic_regulatory_system"
RAW_TABLE     = "demo_projects.default.saibs_raw"
CHUNK_TABLE   = "demo_projects.default.saib_chunks"
EVAL_TABLE    = "demo_projects.default.saib_eval_set"
ENDPOINT      = "saib_vs"
INDEX_NAME    = "demo_projects.default.saib_index"
EMBEDDER      = "databricks-gte-large-en"
LLM           = "databricks-meta-llama-3-3-70b-instruct"

## Step 1 — Parse PDFs

In [0]:

parsed_df = spark.sql(f"""
SELECT _metadata.file_name AS file_name,
       ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
FROM READ_FILES('{VOLUME_PATH}', format => 'binaryFile', fileNamePattern => '*.pdf')
""")

## Step 2 — Extract fields + clean the SAIB ID

In [0]:

import re, json
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, StructType, StructField
from pyspark.sql import functions as F
 
def extract_fields(parsed_json_str):
    try:
        parsed = json.loads(parsed_json_str)
        elements = parsed.get('document', {}).get('elements', [])
        full_text = ''
        for e in elements:
            if e.get('type') in ['text', 'title', 'section_header'] and e.get('content'):
                full_text += e['content'] + '\n'
 
        def grab(pat, flags=re.IGNORECASE):
            m = re.search(pat, full_text, flags)
            return m.group(1).strip() if m else ''
 
        return {
            'saib_number': grab(r'SAIB[:\s]+([\w-]+)'),   # overwritten from filename below
            'subject':     grab(r'SUBJ[:\s]+([^\n]+)') or grab(r'Subject[:\s]+([^\n]+)'),
            'make':        grab(r'Make[:\s]+([^\n]+)'),
            'model':       grab(r'Model[s]?[:\s]+([^\n]+)'),
            'background':      grab(r'Background\s*\n+(.*?)(?=\n\s*(?:Recommendations|Discussion|For Further Information|For Related Service)\s*\n|$)', re.IGNORECASE | re.DOTALL),
            'recommendations': grab(r'Recommendations\s*\n+(.*?)(?=\n\s*(?:For Further Information|For Related Service|Figure)\s*\n|$)', re.IGNORECASE | re.DOTALL),
            'full_text': full_text.strip(),
        }
    except Exception as e:
        return {k: '' for k in ['saib_number', 'subject', 'make', 'model', 'background', 'recommendations']} | {'full_text': f'Error parsing: {e}'}
 
schema = StructType([StructField(c, StringType(), True) for c in
        ['saib_number', 'subject', 'make', 'model', 'background', 'recommendations', 'full_text']])
extract_udf = udf(extract_fields, schema)
 
extracted_df = (parsed_df
    .select('file_name', extract_udf(col('parsed_content').cast('string')).alias('x'))
    .select('file_name', 'x.*'))
 
# SAIB number from filename (uniform source), cleaned in one place for both tables
extracted_df = (extracted_df
    .withColumn('saib_number', F.regexp_replace('file_name', r'\.pdf$', ''))   # strip extension
    .withColumn('saib_number', F.regexp_replace('saib_number', r'%20', ' '))   # URL-decode space
    .withColumn('saib_number', F.regexp_replace('saib_number', r'^DRS-', '')))  # strip source prefix

## Step 3 — Verify extraction

In [0]:
total = extracted_df.count()
failed = extracted_df.filter(col('full_text').startswith('Error parsing')).count()
empty  = extracted_df.filter(col('full_text').isNull() | (F.length('full_text') < 100)).count()
blank_id = extracted_df.filter("saib_number = '' OR saib_number IS NULL").count()
print(f"total={total}  failed={failed}  empty={empty}  blank_id={blank_id}  usable={total - failed - empty}")
 

total=56  failed=0  empty=0  blank_id=0  usable=56


## Step 4 — Write Delta tables

In [0]:

clean_df = extracted_df.filter(~col('full_text').startswith('Error parsing'))
 
# raw structured table
(clean_df.write.format('delta')
    .mode('overwrite').option('overwriteSchema', 'true')
    .saveAsTable(RAW_TABLE))
 
# index source: one chunk per SAIB
chunks = (clean_df
    .select('saib_number', 'subject', F.col('full_text').alias('chunk'))
    .withColumn('id', F.monotonically_increasing_id()))
(chunks.write.format('delta')
    .option('delta.enableChangeDataFeed', 'true')
    .mode('overwrite').option('overwriteSchema', 'true')
    .saveAsTable(CHUNK_TABLE))
 
print("saibs_raw:", spark.table(RAW_TABLE).count(), "| saib_chunks:", spark.table(CHUNK_TABLE).count())

saibs_raw: 56 | saib_chunks: 56


## Step 5 — Vector Search endpoint + index

In [0]:

import time
from databricks.ai_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)
 
# endpoint (first-time provisioning ~15-30 min on a fresh workspace)
if ENDPOINT not in [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]:
    vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")
while vsc.get_endpoint(ENDPOINT)["endpoint_status"]["state"] != "ONLINE":
    print("endpoint provisioning…"); time.sleep(30)
print("endpoint ONLINE")
 
# index
if not any(i["name"] == INDEX_NAME for i in vsc.list_indexes(ENDPOINT).get("vector_indexes", [])):
    vsc.create_delta_sync_index(
        endpoint_name=ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=CHUNK_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="chunk",
        embedding_model_endpoint_name=EMBEDDER)
 
index = vsc.get_index(index_name=INDEX_NAME)
while not index.describe()["status"]["ready"]:
    print("indexing…", index.describe()["status"].get("indexed_row_count", 0)); time.sleep(30)
print("index READY — rows:", index.describe()["status"]["indexed_row_count"])

endpoint ONLINE
index READY — rows: 56


## Step 6 — The agent

In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()
 
SYSTEM = ("You answer questions using ONLY the SAIB documents in the context. "
          "Do not use outside knowledge. Each document is labeled with its SAIB number. "
          "Cite the exact SAIB number(s) you used as 'SAIB <number>'. "
          "If the answer is not in the documents, reply with exactly 'NOT_FOUND' and nothing else.")
 
def rag_answer(query, top_k=3):
    rows = index.similarity_search(
        query_text=query,
        columns=["saib_number", "chunk"],
        num_results=top_k)["result"]["data_array"]
    sources = [r[0] for r in rows]
    context = "\n\n---\n\n".join(f"SAIB {r[0]}:\n{r[1]}" for r in rows)
    resp = w.serving_endpoints.query(
        name=LLM, temperature=0.0,
        messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=SYSTEM),
                  ChatMessage(role=ChatMessageRole.USER,
                              content=f"Context:\n{context}\n\nQuestion: {query}")])
    return {"answer": resp.choices[0].message.content, "sources": sources}

## Step 7 — Evaluation

In [0]:

import pandas as pd
eval_set = [
    ("cracks on EC120B helicopter landing gear skid tubes",                 "AIR-21-09",    "clean",     ""),
    ("incorrect flat washers on Piper wing spar attach bolts",              "AIR-21-07",    "clean",     ""),
    ("rupture disc on evacuation slide activating below 160F",              "AIR-22-06R1",  "clean",     ""),
    ("autopilot randomly swings the nose to north during approach",         "AIR-22-07",    "vague",     ""),
    ("my plane stopped showing up on ATC radar after I powered on avionics","AIR-21-15 R1", "vague",     ""),
    ("helicopter skids cracking after lots of practice landings",           "AIR-21-09",    "vague",     ""),
    ("washers I got don't have the rounded edge the manual shows",          "AIR-21-07",    "vague",     ""),
    ("GPS navigation problem causing wrong heading",                        "AIR-22-07",    "cross-doc", ""),
    ("cracking found on a structural airframe part",                        "2026-15",      "cross-doc", "AIR-21-09"),
    ("ADS-B or transponder not transmitting",                               "AIR-21-15 R1", "cross-doc", ""),
    ("what is the correct tyre pressure for a Cessna 172",                  "NOT_FOUND",    "no-answer", ""),
    ("how often should I change engine oil on a Boeing 737",                "NOT_FOUND",    "no-answer", ""),
    ("what is the maximum takeoff weight of an Airbus A320",                "NOT_FOUND",    "no-answer", ""),
]
(spark.createDataFrame(pd.DataFrame(eval_set, columns=["query", "expected", "category", "also_valid"]))
    .write.format("delta").mode("overwrite").saveAsTable(EVAL_TABLE))
print(spark.table(EVAL_TABLE).count(), "eval rows")

13 eval rows


In [0]:

evdf = spark.table(EVAL_TABLE).toPandas()
rows = []
for _, r in evdf.iterrows():
    out = rag_answer(r["query"], top_k=3)
    valid = {x for x in [r["expected"], (r["also_valid"] or "")] if x}
    ans = out["answer"].strip()
    hit = (len(valid & set(out["sources"])) > 0) if r["category"] != "no-answer" else None
    correct = (ans == "NOT_FOUND") if r["category"] == "no-answer" else (any(v in ans for v in valid) and ans != "NOT_FOUND")
    rows.append({"query": r["query"], "category": r["category"], "sources": out["sources"], "hit": hit, "correct": correct})
res = pd.DataFrame(rows)
 
for cat in ["clean", "vague", "cross-doc", "no-answer"]:
    c = res[res["category"] == cat]
    if cat == "no-answer":
        print(f"{cat:10} refusal: {c['correct'].sum()}/{len(c)}")
    else:
        print(f"{cat:10} retrieval-hit: {c['hit'].sum()}/{len(c)}  answer-correct: {c['correct'].sum()}/{len(c)}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

In [0]:
out = rag_answer("my plane stopped showing up on ATC radar after I powered on the avionics")
print(out["answer"])
print("sources:", out["sources"])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
SAIB AIR-21-15 R1
sources: ['AIR-21-15 R1', '2022-16', 'NM-18-32 R1']
